In [1]:

# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/binary-classification-with-a-bank-dataset-clone/sample_submission.csv
/kaggle/input/binary-classification-with-a-bank-dataset-clone/train.csv
/kaggle/input/binary-classification-with-a-bank-dataset-clone/test.csv


In [2]:
file_path_train = '/kaggle/input/binary-classification-with-a-bank-dataset-clone/train.csv'
train = pd.read_csv(file_path_train)

file_path_test = '/kaggle/input/binary-classification-with-a-bank-dataset-clone/test.csv'
test = pd.read_csv(file_path_test)

file_path_submission = '/kaggle/input/binary-classification-with-a-bank-dataset-clone/sample_submission.csv'
submission = pd.read_csv(file_path_submission)

In [3]:
train.head()

,id,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,0,42,technician,married,secondary,no,7,no,no,cellular,25,aug,117,3,-1,0,unknown,0
1,1,38,blue-collar,married,secondary,no,514,no,no,unknown,18,jun,185,1,-1,0,unknown,0
2,2,36,blue-collar,married,secondary,no,602,yes,no,unknown,14,may,111,2,-1,0,unknown,0
3,3,27,student,single,secondary,no,34,yes,no,unknown,28,may,10,2,-1,0,unknown,0
4,4,26,technician,married,secondary,no,889,yes,no,cellular,3,feb,902,1,-1,0,unknown,1


In [4]:
X = train.drop(columns=['y'])
# X = train.drop(columns=['duration'])
y = train['y']


In [5]:
from sklearn.preprocessing import LabelEncoder

cat_cols = X.select_dtypes(include='object').columns

for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    test[col] = le.transform(test[col])


In [6]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [7]:
from sklearn.ensemble import HistGradientBoostingClassifier

model = HistGradientBoostingClassifier(
    max_depth=6,
    learning_rate=0.05,
    max_iter=200,
    random_state=42
)

model.fit(X_train, y_train)



HistGradientBoostingClassifier(learning_rate=0.05, max_depth=6, max_iter=200,
                               random_state=42)

In [8]:
# val_pred = model.predict_proba(X_val)[:, 1]
# from sklearn.metrics import roc_auc_score
# roc_auc_score(y_val, val_pred)

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import numpy as np

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
aucs = []

for train_idx, val_idx in skf.split(X, y):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = HistGradientBoostingClassifier(
        max_depth=6,
        learning_rate=0.05,
        max_iter=200,
        random_state=42
    )

    model.fit(X_tr, y_tr)
    val_pred = model.predict_proba(X_val)[:, 1]
    aucs.append(roc_auc_score(y_val, val_pred))

print("CV AUCs:", aucs)
print("Mean AUC:", np.mean(aucs))



CV AUCs: [np.float64(0.9636501678989073), np.float64(0.9623299524194731), np.float64(0.9622544996725058), np.float64(0.9635028558819507), np.float64(0.9626939236939958)]
Mean AUC: 0.9628862799133664


In [9]:
model.fit(X, y)


HistGradientBoostingClassifier(learning_rate=0.05, max_depth=6, max_iter=200,
                               random_state=42)

In [10]:
test_pred = model.predict_proba(test)[:, 1]


In [11]:
submission['y'] = test_pred
submission.to_csv('submission.csv', index=False)
print ("successfull")



successfull
